# AI응용 4·5·6·7·8 — 통합 문제풀이

## 목차
1. 4차시 — 추적알고리즘: 템플릿 매칭·MOG2·MeanShift/CamShift·광류·RAFT
2. 5차시 — CNN응용: LeNet·AlexNet·VGG13·합성곱·전이학습
3. 6차시 — 신경망 스타일 전이: Gram Matrix·VGG19·Adam·LBFGS
4. 7차시 — 이미지 생성형 AI: GAN·DCGAN·VAE·U-Net
5. 8차시 — 이미지 캡셔닝: CNN·LSTM·Flickr8k·Embedding·BLEU/METEOR/CIDEr

# 전체 복습 · 차시별 연결

- **4:** 템플릿 매칭, 배경 차분, 색상 추적, Sparse/Dense Flow
- **5:** CNN 특징 추출, LeNet/AlexNet/VGG13, 전이학습
- **6:** 콘텐츠·스타일 손실, Gram Matrix, Adam/LBFGS
- **7:** GAN의 G/D 학습, VAE 재매개변수화, U-Net skip connection
- **8:** CNN 인코더 → LSTM 디코더 → 캡션 생성 → BLEU/METEOR/CIDEr

# 4차시 · 추적알고리즘

**지정 노션 원본:** [상위 학습 페이지](https://app.notion.com/p/41e9ba5c1aae80e1a3a3e6aab71de6e0) · [코드학습 하위 페이지](https://app.notion.com/p/4679ba5c1aae83828427813a31f74797) · [내용 빈칸풀이 하위 페이지](https://app.notion.com/p/45e9ba5c1aae829c9f96810a8f3c1b4c)

## 4-A. 코드 빈칸 문제

각 코드 조각은 **문제용 코드**입니다. `____`를 채우는 연습용으로, 별도 실행에 필요한 이미지·모델·변수는 원본 노션을 참조하세요.

### 코드 4-01 · 템플릿 매칭

```python
result = cv2.____(image_gray, template_gray, cv2.TM_CCOEFF_NORMED)
min_val, max_val, min_loc, max_loc = cv2.____(result)
top_left = ____
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `matchTemplate / minMaxLoc / max_loc`

TM_CCOEFF_NORMED는 최대 점수; TM_SQDIFF 계열은 최소 점수.

**풀이 설명:** 템플릿은 영상의 각 위치에 대조되어 점수맵을 만듭니다. 매칭 방식에 따라 높은 점수 또는 낮은 점수가 더 좋은 일치를 뜻하며, 특징점은 위치와 주변 기술자를 분리해 다룹니다.


</details>

In [ ]:
result = cv2.matchTemplate(image_gray, template_gray, cv2.TM_CCOEFF_NORMED)
min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(result)
top_left = max_loc

### 코드 4-02 · 다중 후보 위치 추출

```python
ys, xs = np.____(result >= threshold)
for x, y in zip(xs, ys):
    cv2.____(image, (x,y), (x+w,y+h), (0,255,0), 2)
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `where / rectangle`

중복 검출 후보에는 NMS 후처리 가능.

**풀이 설명:** 광류는 연속 영상에서의 위치 이동을 추정합니다. Sparse 방식은 선택한 점을, Dense 방식은 영상의 많은 픽셀을 대상으로 하며, 흐름의 두 성분으로 크기와 방향을 계산합니다.


</details>

In [ ]:
ys, xs = np.where(result >= threshold)
for x, y in zip(xs, ys):
    cv2.rectangle(image, (x,y), (x+w,y+h), (0,255,0), 2)

### 코드 4-03 · MOG2 배경 차분

```python
subtractor = cv2.____(history=500, varThreshold=16, detectShadows=True)
fg_mask = subtractor.____(frame)
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `createBackgroundSubtractorMOG2 / apply`

전경 마스크이며 객체 ID는 제공하지 않음.

**풀이 설명:** MOG2는 픽셀의 시간적 색상 분포를 가우시안 혼합으로 모델링해 배경과 전경을 구분합니다. 출력 마스크는 움직임 후보를 나타냅니다.


</details>

In [ ]:
subtractor = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=16, detectShadows=True)
fg_mask = subtractor.apply(frame)

### 코드 4-04 · HSV 히스토그램·역투영

```python
hsv = cv2.____(frame, cv2.COLOR_BGR2HSV)
roi_hist = cv2.____([roi_hsv], [0], mask, [180], [0,180])
cv2.____(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)
back_proj = cv2.____([hsv], [0], roi_hist, [0,180], 1)
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `cvtColor / calcHist / normalize / calcBackProject`

색상 분포를 위치별 점수맵으로 역투영.

**풀이 설명:** 템플릿은 영상의 각 위치에 대조되어 점수맵을 만듭니다. 매칭 방식에 따라 높은 점수 또는 낮은 점수가 더 좋은 일치를 뜻하며, 특징점은 위치와 주변 기술자를 분리해 다룹니다.


</details>

In [ ]:
hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
roi_hist = cv2.calcHist([roi_hsv], [0], mask, [180], [0,180])
cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)
back_proj = cv2.calcBacProject([hsv], [0], roi_hist, [0,180], 1)

### 코드 4-05 · MeanShift와 CamShift

```python
term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1)
_, window = cv2.____(back_proj, track_window, term_crit)
rotated_rect, new_window = cv2.____(back_proj, track_window, term_crit)
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `meanShift / CamShift`

CamShift는 크기·방향에 적응.

**풀이 설명:** HSV 히스토그램은 목표 물체의 색상 분포를 표현하고, 역투영은 현재 프레임의 각 픽셀에 그 분포와의 적합도를 부여합니다. MeanShift는 위치를, CamShift는 크기와 방향까지 조절합니다.


</details>

In [ ]:
term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1)
_, window = cv2.meanShift(back_proj, track_window, term_crit)
rotated_rect, new_window = cv2.CamShift(back_proj, track_window, term_crit)

### 코드 4-06 · Shi–Tomasi 특징점

```python
points = cv2.____(gray, maxCorners=100, qualityLevel=0.3,
                         minDistance=7, blockSize=7)
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `goodFeaturesToTrack`

반환 좌표는 보통 (N,1,2).

**풀이 설명:** 광류는 연속 영상에서의 위치 이동을 추정합니다. Sparse 방식은 선택한 점을, Dense 방식은 영상의 많은 픽셀을 대상으로 하며, 흐름의 두 성분으로 크기와 방향을 계산합니다.


</details>

In [ ]:
points = cv2.goodFeaturesToTrack(gray, maxCorners=100, qualityLevel=0.3,
                         minDistance=7, blockSize=7)

### 코드 4-07 · Sparse Lucas–Kanade

```python
p1, st, err = cv2.____(prev_gray, next_gray, p0, None,
    winSize=(15,15), maxLevel=2, criteria=term_crit)
good_new = p1[st.____() == 1]
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `calcOpticalFlowPyrLK / ravel`

p1: 다음 위치, st: 성공 여부, err: 오차.

**풀이 설명:** 광류는 연속 영상에서의 위치 이동을 추정합니다. Sparse 방식은 선택한 점을, Dense 방식은 영상의 많은 픽셀을 대상으로 하며, 흐름의 두 성분으로 크기와 방향을 계산합니다.


</details>

In [ ]:
p1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, next_gray, p0, None,
    winSize=(15,15), maxLevel=2, criteria=term_crit)
good_new = p1[st.ravel() == 1]

### 코드 4-08 · Dense Farneback

```python
flow = cv2.____(prev_gray, next_gray, None,
    0.5, 3, 15, 3, 5, 1.2, 0)
fx, fy = flow[..., ____], flow[..., ____]
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `calcOpticalFlowFarneback / 0 / 1`

결과 shape=(H,W,2).

**풀이 설명:** 광류는 연속 영상에서의 위치 이동을 추정합니다. Sparse 방식은 선택한 점을, Dense 방식은 영상의 많은 픽셀을 대상으로 하며, 흐름의 두 성분으로 크기와 방향을 계산합니다.


</details>

In [ ]:
flow = cv2.calcOpticalFlowFarneback(prev_gray, next_gray, None,
    0.5, 3, 15, 3, 5, 1.2, 0)
fx, fy = flow[..., 0(x방향)], flow[..., 1(y방향)]

### 코드 4-09 · RAFT 추론

```python
model = raft_small(weights=Raft_Small_Weights.DEFAULT).to(device).____()
with torch.____():
    flow_predictions = model(img1, img2)
final_flow = flow_predictions[____]
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `eval / no_grad / -1`

사전학습 가중치와 입력 전처리 필요.

**풀이 설명:** 광류는 연속 영상에서의 위치 이동을 추정합니다. Sparse 방식은 선택한 점을, Dense 방식은 영상의 많은 픽셀을 대상으로 하며, 흐름의 두 성분으로 크기와 방향을 계산합니다.


</details>

In [ ]:
model = raft_small(weights=Raft_Small_Weights.DEFAULT).to(device).eval()
with torch.no_grad():
    flow_predictions = model(img1, img2)
final_flow = flow_predictions[-1]

## 4-B. 내용 빈칸풀이

### 특징자·템플릿 매칭

**개념 4-01.** 작은 기준 이미지를 원본에서 찾는 방식은?  
**답:**

<details><summary>정답 확인</summary>

템플릿 매칭

**풀이 설명:** 템플릿은 영상의 각 위치에 대조되어 점수맵을 만듭니다. 매칭 방식에 따라 높은 점수 또는 낮은 점수가 더 좋은 일치를 뜻하며, 특징점은 위치와 주변 기술자를 분리해 다룹니다.


</details>

**개념 4-02.** 템플릿 매칭 결과는 위치별 어떤 맵인가?  
**답:**

<details><summary>정답 확인</summary>

유사도 또는 매칭 점수

**풀이 설명:** 템플릿은 영상의 각 위치에 대조되어 점수맵을 만듭니다. 매칭 방식에 따라 높은 점수 또는 낮은 점수가 더 좋은 일치를 뜻하며, 특징점은 위치와 주변 기술자를 분리해 다룹니다.


</details>

**개념 4-03.** TM_CCOEFF_NORMED는 최대/최소 어느 점수를 선택하는가?  
**답:**

<details><summary>정답 확인</summary>

최댓값

**풀이 설명:** 템플릿은 영상의 각 위치에 대조되어 점수맵을 만듭니다. 매칭 방식에 따라 높은 점수 또는 낮은 점수가 더 좋은 일치를 뜻하며, 특징점은 위치와 주변 기술자를 분리해 다룹니다.


</details>

**개념 4-04.** Keypoint와 Descriptor는 각각 무엇을 표현하는가?  
**답:**

<details><summary>정답 확인</summary>

특징의 위치 / 특징 기술 벡터

**풀이 설명:** 템플릿은 영상의 각 위치에 대조되어 점수맵을 만듭니다. 매칭 방식에 따라 높은 점수 또는 낮은 점수가 더 좋은 일치를 뜻하며, 특징점은 위치와 주변 기술자를 분리해 다룹니다.


</details>


### 풀이
1 :  탬플릿 매칭
2 :  유사도 맵
3 :  최대값
4 : 특징의 위치/ 위치에서의 벡터

### 배경 차분·색상 추적

**개념 4-05.** MOG2가 출력하는 마스크는?  
**답:**

<details><summary>정답 확인</summary>

전경

**풀이 설명:** MOG2는 픽셀의 시간적 색상 분포를 가우시안 혼합으로 모델링해 배경과 전경을 구분합니다. 출력 마스크는 움직임 후보를 나타냅니다.


</details>

**개념 4-06.** 색상 추적에 쓰기 쉬운 색 공간은?  
**답:**

<details><summary>정답 확인</summary>

HSV

**풀이 설명:** HSV 히스토그램은 목표 물체의 색상 분포를 표현하고, 역투영은 현재 프레임의 각 픽셀에 그 분포와의 적합도를 부여합니다. MeanShift는 위치를, CamShift는 크기와 방향까지 조절합니다.


</details>

**개념 4-07.** 히스토그램을 이미지 위치별 점수로 되돌리는 연산은?  
**답:**

<details><summary>정답 확인</summary>

역투영(Back Projection)

**풀이 설명:** 템플릿은 영상의 각 위치에 대조되어 점수맵을 만듭니다. 매칭 방식에 따라 높은 점수 또는 낮은 점수가 더 좋은 일치를 뜻하며, 특징점은 위치와 주변 기술자를 분리해 다룹니다.


</details>

**개념 4-08.** MeanShift와 CamShift의 추적 윈도우 차이는?  
**답:**

<details><summary>정답 확인</summary>

MeanShift는 크기 유지 / CamShift는 크기·방향 적응

**풀이 설명:** HSV 히스토그램은 목표 물체의 색상 분포를 표현하고, 역투영은 현재 프레임의 각 픽셀에 그 분포와의 적합도를 부여합니다. MeanShift는 위치를, CamShift는 크기와 방향까지 조절합니다.


</details>


### 풀이
5 : 전경
6 : HSV
7 : 역투영
8 :  MeanShift - 크기고정 / CamShift - 크기,방향 회전

### Lucas–Kanade 광류

**개념 4-09.** Optical Flow는 연속 프레임 사이 무엇을 추정하는가?  
**답:**

<details><summary>정답 확인</summary>

픽셀 또는 특징점의 이동

**풀이 설명:** 광류는 연속 영상에서의 위치 이동을 추정합니다. Sparse 방식은 선택한 점을, Dense 방식은 영상의 많은 픽셀을 대상으로 하며, 흐름의 두 성분으로 크기와 방향을 계산합니다.


</details>

**개념 4-10.** Sparse LK에서 이전 특징점 입력 변수는?  
**답:**

<details><summary>정답 확인</summary>

p0

**풀이 설명:** 광류는 연속 영상에서의 위치 이동을 추정합니다. Sparse 방식은 선택한 점을, Dense 방식은 영상의 많은 픽셀을 대상으로 하며, 흐름의 두 성분으로 크기와 방향을 계산합니다.


</details>

**개념 4-11.** p1, st, err의 의미는?  
**답:**

<details><summary>정답 확인</summary>

다음 위치 / 추적 성공 상태 / 추적 오차

**풀이 설명:** 이 문제는 추적 알고리즘의 입력·출력 및 함수 역할을 묻습니다. 해당 연산이 처리하는 영상 데이터와 반환 결과를 함께 확인하세요.


</details>

**개념 4-12.** LK의 대표 가정 3가지는?  
**답:**

<details><summary>정답 확인</summary>

밝기 일정성 / 작은 움직임 / 주변 픽셀의 유사한 움직임

**풀이 설명:** 이 문제는 추적 알고리즘의 입력·출력 및 함수 역할을 묻습니다. 해당 연산이 처리하는 영상 데이터와 반환 결과를 함께 확인하세요.


</details>


### 풀이
9 : 특징점의 이동과 픽셀
10 : prevPt (초기값은 cv2.goodFeaturesToTrack결과)
11 : cv2.calcOpticalFlwPyrLK의 출력 -
p1 : 현재 프레임에서 추정한 점 좌표,
st : 추적 성공 여부,
err : 추적 오차
12 : 밝기보정, 작은이동, 국소 영역

### Dense Flow·RAFT

**개념 4-13.** Sparse와 Dense 광류가 추정하는 대상의 차이는?  
**답:**

<details><summary>정답 확인</summary>

선택 특징점 / 모든 픽셀

**풀이 설명:** 광류는 연속 영상에서의 위치 이동을 추정합니다. Sparse 방식은 선택한 점을, Dense 방식은 영상의 많은 픽셀을 대상으로 하며, 흐름의 두 성분으로 크기와 방향을 계산합니다.


</details>

**개념 4-14.** Farneback 흐름 텐서의 shape는?  
**답:**

<details><summary>정답 확인</summary>

(H,W,2)

**풀이 설명:** 광류는 연속 영상에서의 위치 이동을 추정합니다. Sparse 방식은 선택한 점을, Dense 방식은 영상의 많은 픽셀을 대상으로 하며, 흐름의 두 성분으로 크기와 방향을 계산합니다.


</details>

**개념 4-15.** flow[...,0]과 flow[...,1]의 의미는?  
**답:**

<details><summary>정답 확인</summary>

x(수평) / y(수직)

**풀이 설명:** 광류는 연속 영상에서의 위치 이동을 추정합니다. Sparse 방식은 선택한 점을, Dense 방식은 영상의 많은 픽셀을 대상으로 하며, 흐름의 두 성분으로 크기와 방향을 계산합니다.


</details>

**개념 4-16.** RAFT의 All-Pairs 구성 요소는?  
**답:**

<details><summary>정답 확인</summary>

Correlation(상관관계)

**풀이 설명:** 광류는 연속 영상에서의 위치 이동을 추정합니다. Sparse 방식은 선택한 점을, Dense 방식은 영상의 많은 픽셀을 대상으로 하며, 흐름의 두 성분으로 크기와 방향을 계산합니다.


</details>

**개념 4-17.** RAFT는 추정치를 어떻게 정제하는가?  
**답:**

<details><summary>정답 확인</summary>

반복적으로

**풀이 설명:** 광류는 연속 영상에서의 위치 이동을 추정합니다. Sparse 방식은 선택한 점을, Dense 방식은 영상의 많은 픽셀을 대상으로 하며, 흐름의 두 성분으로 크기와 방향을 계산합니다.


</details>


### 풀이
13 : cv2.calcOptcalFlpwFarneback.shape == (H,W,2)
14 : (H,W,2)
15 : flow[0] : x방향, flow[1] : y방향
16 : 상관 관계
17 : 반복적으로 정제하여 정확도를 높인다

# 5차시 · CNN응용

**지정 노션 원본:** [상위 학습 페이지](https://app.notion.com/p/41e9ba5c1aae8050b557ebec940ea704) · [코드학습 하위 페이지](https://app.notion.com/p/41e9ba5c1aae80b9a1d0ed9a6e6a30b3) · [내용 빈칸풀이 하위 페이지](https://app.notion.com/p/41e9ba5c1aae80fb828edd80150c6cd6)

## 5-A. 코드 빈칸 문제

### 코드 5 · LeNet

```python
class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        self.cn1 = nn.Conv2d(__, __, 5)       # RGB 3 → 특징맵 6
        self.cn2 = nn.Conv2d(__, __, 5)      # 특징맵 6 → 16
        self.fc1 = nn.Linear(__, 120)   # 평탄화 400 → 120
        self.fc2 = nn.Linear(120, 84)       # 120 → 84
        self.fc3 = nn.Linear(84, 10)        # 최종 클래스 10
        
    def forward(self, x):
        x = F.____(self.cn1(x))             # 첫 합성곱·활성화
        x = F.______(x, (2, 2))        # 공간 크기 절반
        x = F.____(self.cn2(x))             # 둘째 합성곱·활성화
        x = F.______(x, (2, 2))
        x = x.view(__, self.flattened_features(x))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.___(x)                  # logits 반환

    def flattened_features(self, x):
        num_feats = 1
        for size in x.size()[1:]:
            num_feats *= size
        return num_feats
```

### 코드 5-01 · Conv2d 채널

```python
# nn.Conv2d(입력채널 수, 출력채널 수, 커널크기)
self.cn1 = nn.Conv2d(3, 6, 5) # RGB 3 → 특징맵 6
self.cn2 = nn.Conv2d(6, 16, 5) # 특징맵 6 → 16
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `3 / 6 / 6 / 16`

출력 채널 수는 필터 개수.

**풀이 설명:** 합성곱의 입력 채널 수는 이전 층의 출력 채널과 일치해야 합니다. 출력 채널 수는 필터 개수로 결정되며, 커널 크기·패딩·스트라이드가 공간 크기를 바꿉니다.


</details>

### 코드 5-02 · LeNet FC·Flatten

```python
self.fc1 = nn.Linear(16*5*5, 120) # (16,5,5) → 400
x = x.view(-1, self.flattened_features(x))
return self.fc3(x) # 최종 10개 클래스 점수
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `16*5*5 / -1 / fc3`

입력 32×32 기준 flatten 400.

**풀이 설명:** 합성곱 특징맵을 완전연결층에 전달하려면 배치 차원은 보존하고 나머지 차원을 평탄화합니다. Linear의 in_features는 평탄화 결과의 요소 수와 일치해야 합니다.


</details>

### 코드 5-03 · Pooling·활성화

```python
x = F.relu(self.cn1(x))
x = F.max_pool2d(x, (2, 2))
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `relu / max_pool2d`

ReLU 후 MaxPool.

**풀이 설명:** Pooling은 공간 크기를 줄여 중요한 정보를 요약하고, ReLU는 음수 입력을 0으로 바꾸는 비선형성을 부여합니다.


</details>

### 코드 5-04 · AlexNet 데이터 전처리

```python
transforms.Compose([
    transforms.RandomResizedCrop(____), # 학습 이미지 증강
    transforms.RandomHorizontalFlip(),
    transforms.____(), # 텐서 변환
    transforms.Normalize([.490,.449,.411],[.231,.221,.230])
])
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `224 / ToTensor`

검증 데이터는 Resize와 CenterCrop.

**풀이 설명:** 전이학습에서는 사전학습 특징 추출기를 재사용하고 마지막 분류층을 새 데이터의 클래스 수에 맞게 교체합니다. 평가 시에는 eval과 no_grad를 사용합니다.


</details>

In [ ]:
transforms.Compose([
    transforms.RandomResizedCrop(224), # 학습 이미지 증강
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), # 텐서 변환
    transforms.Normalize([.490,.449,.411],[.231,.221,.230])
])

### 코드 5-05 · ImageFolder·DataLoader

```python
dataset = datasets.____(folder, transform)
loader = torch.utils.data.____(dataset, batch_size=8, shuffle=True)
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `ImageFolder / DataLoader`

클래스별 폴더 및 미니배치.

**풀이 설명:** ImageFolder는 하위 폴더 이름을 클래스 라벨로 읽습니다. transform은 입력 크기와 정규화를 맞추고 DataLoader는 미니배치를 생성합니다.


</details>

In [ ]:
dataset = datasets.ImageFolder(folder, transform)
loader = torch.utils.data.DataLoader(dataset, batch_size = 8, shuffle=True)

### 코드 5-06 · 학습 루프 순서

```python
optimizer.____()
outputs = model(images)
loss = loss_func(outputs, labels)
loss.____()
optimizer.____()
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `zero_grad / backward / step`

기울기 초기화 → 순전파 → 손실 → 역전파 → 갱신.

**풀이 설명:** 학습 단계는 기울기 초기화 → 순전파 → 손실 계산 → 역전파 → 파라미터 갱신 순서입니다. zero_grad를 빼면 이전 기울기가 누적됩니다.


</details>

In [ ]:
optimizer.zero_grad()
outputs = model(images)
loss = loss_func(outputs, labels)
loss.backward()
optimizer.step()

### 코드 5-07 · 분류기 교체·손실

```python
model = models.alexnet(pretrained=True)
model.classifier[____] = nn.Linear(4096, ____)
criterion = nn.____()
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `6 / 2 / CrossEntropyLoss`

개미·벌 2클래스. 최신 torchvision은 weights 인수 권장.

**풀이 설명:** 전이학습에서는 사전학습 특징 추출기를 재사용하고 마지막 분류층을 새 데이터의 클래스 수에 맞게 교체합니다. 평가 시에는 eval과 no_grad를 사용합니다.


</details>

In [ ]:
model = models.alexnet(pretrained=True)
model.classifer[6] = nn.Linear(4096,2)
criterion = nn.CrossEntropyLoss()

### 코드 5-08 · 평가 모드·추론

```python
model.____()
with torch.____():
    logits = model(images)
    preds = logits.____(dim=1)
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `eval / no_grad / argmax`

VGG13 ImageNet 1000클래스 추론.

**풀이 설명:** Pooling은 공간 크기를 줄여 중요한 정보를 요약하고, ReLU는 음수 입력을 0으로 바꾸는 비선형성을 부여합니다.


</details>

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(images)
    preds = logits.argmax(dim=1)

### 코드 5-09 · OpenCV 색상·좌표

```python
rgb = cv2.____(bgr, cv2.COLOR_BGR2RGB)
result = cv2.____(img, map_x, map_y, interpolation=cv2.INTER_CUBIC)
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `cvtColor / remap`

좌표 지도에 따른 픽셀 재배치.

**풀이 설명:** 이 문제는 CNN 모델의 텐서 형상 또는 학습·평가 과정의 역할을 확인합니다. 앞뒤 층의 입력과 출력이 연결되는지 검토하세요.


</details>

In [ ]:
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
result = cv2.remap(img, map_x, map_y, interpolation=cv2.INTER_CUBIC)

## 5-B. 내용 빈칸풀이

### CNN 기초·합성곱

**개념 5-01.** 픽셀값의 행·열 배열은?  
**답:**

<details><summary>정답 확인</summary>

Matrix(행렬)

**풀이 설명:** 이 문제는 CNN 모델의 텐서 형상 또는 학습·평가 과정의 역할을 확인합니다. 앞뒤 층의 입력과 출력이 연결되는지 검토하세요.


</details>

**개념 5-02.** RGB 채널 수는?  
**답:**

<details><summary>정답 확인</summary>

3

**풀이 설명:** 합성곱의 입력 채널 수는 이전 층의 출력 채널과 일치해야 합니다. 출력 채널 수는 필터 개수로 결정되며, 커널 크기·패딩·스트라이드가 공간 크기를 바꿉니다.


</details>

**개념 5-03.** 동일 필터 가중치를 여러 위치에 적용하는 특성은?  
**답:**

<details><summary>정답 확인</summary>

가중치 공유

**풀이 설명:** 합성곱의 입력 채널 수는 이전 층의 출력 채널과 일치해야 합니다. 출력 채널 수는 필터 개수로 결정되며, 커널 크기·패딩·스트라이드가 공간 크기를 바꿉니다.


</details>

**개념 5-04.** Stride와 Padding의 의미는?  
**답:**

<details><summary>정답 확인</summary>

필터 이동 간격 / 이미지 경계 확장

**풀이 설명:** 합성곱의 입력 채널 수는 이전 층의 출력 채널과 일치해야 합니다. 출력 채널 수는 필터 개수로 결정되며, 커널 크기·패딩·스트라이드가 공간 크기를 바꿉니다.


</details>

**개념 5-05.** W=32,K=5,P=0,S=1의 출력 폭은?  
**답:**

<details><summary>정답 확인</summary>

28

**풀이 설명:** 이 문제는 CNN 모델의 텐서 형상 또는 학습·평가 과정의 역할을 확인합니다. 앞뒤 층의 입력과 출력이 연결되는지 검토하세요.


</details>


### 풀이
1 : 2차원 행렬
2 : 3
3 : 가중치 공유(동일 필터를 여러 공간 위치에 적용하여 매개 변수 줄이기)
4 : Stride = 필터 이동  간격, Padding : 이미지 빈칸 채우기
5 : (32+2*0-5)/1 + 1 = 28

### Pooling·분류기

**개념 5-06.** Max/Average Pooling의 집계 방식은?  
**답:**

<details><summary>정답 확인</summary>

최댓값 / 평균

**풀이 설명:** Pooling은 공간 크기를 줄여 중요한 정보를 요약하고, ReLU는 음수 입력을 0으로 바꾸는 비선형성을 부여합니다.


</details>

**개념 5-07.** 특징맵을 FC 입력 벡터로 펼치는 연산은?  
**답:**

<details><summary>정답 확인</summary>

Flatten

**풀이 설명:** 합성곱의 입력 채널 수는 이전 층의 출력 채널과 일치해야 합니다. 출력 채널 수는 필터 개수로 결정되며, 커널 크기·패딩·스트라이드가 공간 크기를 바꿉니다.


</details>

**개념 5-08.** 클래스 점수를 합이 1인 값으로 변환하는 함수는?  
**답:**

<details><summary>정답 확인</summary>

Softmax

**풀이 설명:** Pooling은 공간 크기를 줄여 중요한 정보를 요약하고, ReLU는 음수 입력을 0으로 바꾸는 비선형성을 부여합니다.


</details>

**개념 5-09.** nn.Conv2d(3,6,5)의 출력 채널은?  
**답:**

<details><summary>정답 확인</summary>

6

**풀이 설명:** 합성곱의 입력 채널 수는 이전 층의 출력 채널과 일치해야 합니다. 출력 채널 수는 필터 개수로 결정되며, 커널 크기·패딩·스트라이드가 공간 크기를 바꿉니다.


</details>


### 풀이
6 : Max : 영역 최대값 선택, Average : 영역 평균 계산
7 : Flatten()
8 : Softmax
9 : 6개

### LeNet·AlexNet·VGG13

**개념 5-10.** LeNet 실습의 출력 클래스 수는?  
**답:**

<details><summary>정답 확인</summary>

10

**풀이 설명:** 합성곱 특징맵을 완전연결층에 전달하려면 배치 차원은 보존하고 나머지 차원을 평탄화합니다. Linear의 in_features는 평탄화 결과의 요소 수와 일치해야 합니다.


</details>

**개념 5-11.** AlexNet 실습의 출력 수와 클래스는?  
**답:**

<details><summary>정답 확인</summary>

2 / ants(개미) / bees(벌)

**풀이 설명:** 전이학습에서는 사전학습 특징 추출기를 재사용하고 마지막 분류층을 새 데이터의 클래스 수에 맞게 교체합니다. 평가 시에는 eval과 no_grad를 사용합니다.


</details>

**개념 5-12.** AlexNet 손실함수는?  
**답:**

<details><summary>정답 확인</summary>

CrossEntropyLoss

**풀이 설명:** 전이학습에서는 사전학습 특징 추출기를 재사용하고 마지막 분류층을 새 데이터의 클래스 수에 맞게 교체합니다. 평가 시에는 eval과 no_grad를 사용합니다.


</details>

**개념 5-13.** 학습·평가 모드 전환 메서드는?  
**답:**

<details><summary>정답 확인</summary>

train() / eval()

**풀이 설명:** 학습 단계는 기울기 초기화 → 순전파 → 손실 계산 → 역전파 → 파라미터 갱신 순서입니다. zero_grad를 빼면 이전 기울기가 누적됩니다.


</details>

**개념 5-14.** VGG13 사전학습 ImageNet 클래스 수는?  
**답:**

<details><summary>정답 확인</summary>

1000

**풀이 설명:** 전이학습에서는 사전학습 특징 추출기를 재사용하고 마지막 분류층을 새 데이터의 클래스 수에 맞게 교체합니다. 평가 시에는 eval과 no_grad를 사용합니다.


</details>

### 풀이
10 : self.fc = nn.Linear(84,10), 출력 클래스 : 10개
11 : model_finetuned.classifier[6] = nn.Linear(4096, 2) : 2개 (개미,벌)
12 : nn.CrossEntropyLoss() (반환 : logit, 정답인덱스)
13 : model.train(), model.eval()
14 : Imagenet1000_clsidx_tolabels.txt : 클래스 1000개

### 데이터·OpenCV

**개념 5-16.** 클래스별 이미지 폴더를 읽는 데이터셋은?  
**답:**

<details><summary>정답 확인</summary>

ImageFolder

**풀이 설명:** ImageFolder는 하위 폴더 이름을 클래스 라벨로 읽습니다. transform은 입력 크기와 정규화를 맞추고 DataLoader는 미니배치를 생성합니다.


</details>

**개념 5-17.** 미니배치와 셔플 담당 객체는?  
**답:**

<details><summary>정답 확인</summary>

DataLoader

**풀이 설명:** ImageFolder는 하위 폴더 이름을 클래스 라벨로 읽습니다. transform은 입력 크기와 정규화를 맞추고 DataLoader는 미니배치를 생성합니다.


</details>

**개념 5-18.** BGR→RGB 변환 함수는?  
**답:**

<details><summary>정답 확인</summary>

cv2.cvtColor

**풀이 설명:** 이 문제는 CNN 모델의 텐서 형상 또는 학습·평가 과정의 역할을 확인합니다. 앞뒤 층의 입력과 출력이 연결되는지 검토하세요.


</details>

**개념 5-19.** 좌표 맵에 따른 픽셀 재배치 함수는?  
**답:**

<details><summary>정답 확인</summary>

cv2.remap

**풀이 설명:** 이 문제는 CNN 모델의 텐서 형상 또는 학습·평가 과정의 역할을 확인합니다. 앞뒤 층의 입력과 출력이 연결되는지 검토하세요.


</details>

**개념 5-20.** AlexNet과 VGG13 실습의 목적 차이는?  
**답:**

<details><summary>정답 확인</summary>

전이학습/미세조정 / 사전학습 추론

**풀이 설명:** 전이학습에서는 사전학습 특징 추출기를 재사용하고 마지막 분류층을 새 데이터의 클래스 수에 맞게 교체합니다. 평가 시에는 eval과 no_grad를 사용합니다.


</details>


### 풀이
16 : ImageFolder
17 : torch.utils.data.DataLoader(img_data['train'],batch_size=8,shuffle=True,num_workers=2) : DataLoader
18 : cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
19 : cv2.remap(img, map_x,map_y, interpolation=CV2.INTER_CUBIC, borderMode = cv2.BORDER_REFLECT)
20 :
AlexNet - 추출기 재사용 후 분류기 클래스를 2개로 수정하고 손실함수 최적화를 적용하였다. 일부 추출기 학습이 이뤄지는 전이 학습으로 미세조정으로 사용하였다.

VGG13 - 새로운 데이터에 맡는 특징 추출기 전체 모델 학습을 수행하였다.

# 6차시 · 신경망 스타일 전이

**지정 노션 원본:** [상위 학습 페이지](https://app.notion.com/p/41e9ba5c1aae8003a638dfdf6bbfb6ff) · [코드학습 하위 페이지](https://app.notion.com/p/6b79ba5c1aae82b4a13c81e71338599f) · [내용 빈칸풀이 하위 페이지](https://app.notion.com/p/a5f9ba5c1aae82948c2301a7a8b0b2f0)

## 6-A. 코드 빈칸 문제


### 코드 6-01 · Gram Matrix shape

```python
b,c,h,w = x.size()
features = x.reshape(b, c, ____)
gram = torch.____(features, features.transpose(1,2))
return gram / (____)
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `h*w / bmm / h*w`

(B,C,H,W) → (B,C,C); 노션 첫 실습 정규화.

**풀이 설명:** Gram Matrix는 채널별 특징 사이의 상관관계를 계산합니다. 공간 축을 평탄화해 채널×채널 행렬을 만들고 스타일 통계의 차이를 손실로 사용합니다.


</details>

In [ ]:
b,c,h,w = x.size()
features = x.reshape(b, c, h*w) # H와 W만 합쳐 공간 차원을 평탄화: (B,C,N)
gram = torch.bmm(features, features.transpose(1,2))  # (B,C,N) @ (B,N,C) → (B,C,C) : 내적결과
return gram / (h*w) # 공간 크기로 정규화

### 코드 6-02 · 스타일·콘텐츠 손실

```python
style_loss = F.____(gram_generated, gram_style)
content_loss = F.____(features_generated, features_content)
total_loss = style_weight * style_loss + content_weight * content_loss
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `mse_loss / mse_loss`

스타일은 Gram, 콘텐츠는 특징맵 비교.

**풀이 설명:** Gram Matrix는 채널별 특징 사이의 상관관계를 계산합니다. 공간 축을 평탄화해 채널×채널 행렬을 만들고 스타일 통계의 차이를 손실로 사용합니다.


</details>

In [ ]:
import torch.nn.functional as F # 함수형 API 로 객체 생성 불필요
import torch.nn as nn # 클래스형 API로 객체 생성 필요 ex) criterion = nn.MSELoss()

gram_generated = GramMatrix()(input) # input: 생성 이미지의 VGG 특징맵 (B,C,H,W)
style_loss = F.mse_loss(gram_generated, gram_style) # gram_style : 스타일 이미지의 목표 GramMarix
content_loss =F.mse_loss(features_generated, features_content)
total_loss = style_weight * style_loss + content_weight * content_loss

### 코드 6-03 · VGG19 특징 추출기 고정

```python
for p in vgg.parameters():
    p.____ = False
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `requires_grad`

생성 이미지 픽셀을 최적화.

**풀이 설명:** 사전학습 VGG 특징 추출기의 가중치는 고정하고, 생성 이미지의 픽셀을 최적화 대상으로 두어야 원하는 스타일 전이가 일어납니다.


</details>

In [ ]:
for p in vgg.parameters():
    p.requires_grad = False # 속성 직접 설정방식

### 코드 6-04 · 생성 이미지 초기화

```python
generated = content_img.____().____().requires_grad_(True)
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `clone / detach`

독립된 최적화 대상으로 복사.

**풀이 설명:** 전처리에서 크기와 채널 정규화는 특징 추출기 학습 조건과 일치해야 합니다. 콘텐츠·스타일·생성 이미지의 텐서 형상도 호환되어야 합니다.


</details>

In [ ]:
generated = content_img.detach().clone().requires_grad_(True)

### 코드 6-05 · 이미지 정규화

```python
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.____(),
    transforms.____(mean=[.485,.456,.406],std=[.229,.224,.225])
])
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `ToTensor / Normalize`

역정규화 시 동일 mean/std.

**풀이 설명:** 전처리에서 크기와 채널 정규화는 특징 추출기 학습 조건과 일치해야 합니다. 콘텐츠·스타일·생성 이미지의 텐서 형상도 호환되어야 합니다.


</details>

In [ ]:
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.485,0.456,0.406],
        std = [0.229, 0.224, 0.225]
    )

])

### 코드 6-06 · Adam 순서

```python
optimizer.____()
features = vgg(generated)
loss = calculate_loss(features)
loss.____()
optimizer.____()
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `zero_grad / backward / step`

calculate_loss는 노션의 가상 함수.

**풀이 설명:** Adam은 일반적인 zero_grad → loss → backward → step 흐름을 사용합니다. LBFGS는 내부적으로 손실·기울기를 반복 평가하므로 closure 함수가 필요합니다.


</details>

In [ ]:
optmizer.zero_grad()
features = vgg(generated)
loss = calculate_loss(features)
loss.backward()
optimizer.step()

### 코드 6-07 · LBFGS closure

```python
def closure():
    optimizer.____()
    loss = calculate_loss(vgg(generated))
    loss.____()
    return loss
optimizer.step(____)
```

<details><summary>정답·해설 펼치기</summary>

**정답:** `zero_grad / backward / closure`

LBFGS는 closure를 여러 번 호출할 수 있음.

**풀이 설명:** Adam은 일반적인 zero_grad → loss → backward → step 흐름을 사용합니다. LBFGS는 내부적으로 손실·기울기를 반복 평가하므로 closure 함수가 필요합니다.


</details>

In [ ]:
def closure():
    optimizer.zero_grad()
    loss = calculate_loss(vgg(generated))
    # 생성 이미지의 특징맵 추출 + 스타일 + 콘텐츠 손실 가중합
    loss.backward()
    return loss
optimizer.step(closure)

## 6-B. 내용 빈칸풀이

### NST 개요·특징맵

**개념 6-01.** NST가 결합하는 두 이미지의 역할은?  
**답:** `____________`

<details><summary>정답 확인</summary>

콘텐츠 구조 / 스타일 질감

**풀이 설명:** Gram Matrix는 채널별 특징 사이의 상관관계를 계산합니다. 공간 축을 평탄화해 채널×채널 행렬을 만들고 스타일 통계의 차이를 손실로 사용합니다.


</details>

**개념 6-02.** VGG19의 역할은?  
**답:** `____________`

<details><summary>정답 확인</summary>

고정 특징 추출기

**풀이 설명:** 사전학습 VGG 특징 추출기의 가중치는 고정하고, 생성 이미지의 픽셀을 최적화 대상으로 두어야 원하는 스타일 전이가 일어납니다.


</details>

**개념 6-03.** NST의 최적화 대상은?  
**답:** `____________`

<details><summary>정답 확인</summary>

생성 이미지 픽셀값

**풀이 설명:** 전처리에서 크기와 채널 정규화는 특징 추출기 학습 조건과 일치해야 합니다. 콘텐츠·스타일·생성 이미지의 텐서 형상도 호환되어야 합니다.


</details>

**개념 6-04.** PyTorch 이미지 텐서 차원 순서는?  
**답:** `____________`

<details><summary>정답 확인</summary>

(B,C,H,W)

**풀이 설명:** 전처리에서 크기와 채널 정규화는 특징 추출기 학습 조건과 일치해야 합니다. 콘텐츠·스타일·생성 이미지의 텐서 형상도 호환되어야 합니다.


</details>


### 풀이
1 : 콘텐츠 이미지 : 이미지 구조, 스타일 이미지 : 스타일 질감
2 : 특징 추출기
3 : 생성된 이미지의 픽셀값
4 : BCHW

### Gram Matrix·손실

**개념 6-05.** 특징맵 (B,C,H,W)의 Gram 출력 shape는?  
**답:** `____________`

<details><summary>정답 확인</summary>

(B,C,C)

**풀이 설명:** Gram Matrix는 채널별 특징 사이의 상관관계를 계산합니다. 공간 축을 평탄화해 채널×채널 행렬을 만들고 스타일 통계의 차이를 손실로 사용합니다.


</details>

**개념 6-06.** Gram Matrix는 무엇의 공동 활성화를 요약하는가?  
**답:** `____________`

<details><summary>정답 확인</summary>

채널

**풀이 설명:** Gram Matrix는 채널별 특징 사이의 상관관계를 계산합니다. 공간 축을 평탄화해 채널×채널 행렬을 만들고 스타일 통계의 차이를 손실로 사용합니다.


</details>

**개념 6-07.** 콘텐츠 손실 비교 대상은?  
**답:** `____________`

<details><summary>정답 확인</summary>

VGG 특징맵

**풀이 설명:** Gram Matrix는 채널별 특징 사이의 상관관계를 계산합니다. 공간 축을 평탄화해 채널×채널 행렬을 만들고 스타일 통계의 차이를 손실로 사용합니다.


</details>

**개념 6-08.** 스타일 손실 비교 대상은?  
**답:** `____________`

<details><summary>정답 확인</summary>

Gram Matrix

**풀이 설명:** Gram Matrix는 채널별 특징 사이의 상관관계를 계산합니다. 공간 축을 평탄화해 채널×채널 행렬을 만들고 스타일 통계의 차이를 손실로 사용합니다.


</details>

**개념 6-09.** 두 실습의 기본 손실은?  
**답:** `____________`

<details><summary>정답 확인</summary>

MSE(평균제곱오차)

**풀이 설명:** 콘텐츠 손실은 생성 이미지와 콘텐츠 이미지의 특징 차이를, 스타일 손실은 Gram Matrix 차이를 측정합니다. 두 항에 가중치를 주어 최적화합니다.


</details>


### 풀이
5 : B,C,HxW
6 : HxW(=채널)
7 : VGG특징맵
8 : GramMatrix
9 : MSE

### 전처리·학습 대상

**개념 6-10.** 픽셀을 텐서로 바꾸는 함수는?  
**답:** `____________`

<details><summary>정답 확인</summary>

ToTensor

**풀이 설명:** 전처리에서 크기와 채널 정규화는 특징 추출기 학습 조건과 일치해야 합니다. 콘텐츠·스타일·생성 이미지의 텐서 형상도 호환되어야 합니다.


</details>

**개념 6-11.** 채널별 평균/표준편차 정규화 함수는?  
**답:** `____________`

<details><summary>정답 확인</summary>

Normalize

**풀이 설명:** 전처리에서 크기와 채널 정규화는 특징 추출기 학습 조건과 일치해야 합니다. 콘텐츠·스타일·생성 이미지의 텐서 형상도 호환되어야 합니다.


</details>

**개념 6-12.** 역정규화 std의 두 번째 값이 0.244로 바뀌면 어느 채널이 달라지는가?  
**답:** `____________`

<details><summary>정답 확인</summary>

녹색(G)

**풀이 설명:** 전처리에서 크기와 채널 정규화는 특징 추출기 학습 조건과 일치해야 합니다. 콘텐츠·스타일·생성 이미지의 텐서 형상도 호환되어야 합니다.


</details>

**개념 6-13.** 생성 이미지 기울기 활성화 메서드는?  
**답:** `____________`

<details><summary>정답 확인</summary>

requires_grad_(True)

**풀이 설명:** 사전학습 VGG 특징 추출기의 가중치는 고정하고, 생성 이미지의 픽셀을 최적화 대상으로 두어야 원하는 스타일 전이가 일어납니다.


</details>

**개념 6-14.** 콘텐츠 이미지와 계산 그래프 분리 메서드는?  
**답:** `____________`

<details><summary>정답 확인</summary>

detach()

**풀이 설명:** 콘텐츠 손실은 생성 이미지와 콘텐츠 이미지의 특징 차이를, 스타일 손실은 Gram Matrix 차이를 측정합니다. 두 항에 가중치를 주어 최적화합니다.


</details>


### 풀이
10 : torchvision.transforms.ToTensor()
11 : torchvision.transforms.Normalize(mean, std)
Normalize
12 : green(*색상 채널 순서 - opencv : BGR, PyTorch, : RGB)
13 : p.requires_grad_(True)
(* 속성 직접 설정 p.requires_grad = True)
14 : detach()

### 최적화·검증

**개념 6-15.** Adam 루프의 손실 계산 전·후 단계는?  
**답:** `____________`

<details><summary>정답 확인</summary>

특징 추출(순전파) / backward(역전파)

**풀이 설명:** 콘텐츠 손실은 생성 이미지와 콘텐츠 이미지의 특징 차이를, 스타일 손실은 Gram Matrix 차이를 측정합니다. 두 항에 가중치를 주어 최적화합니다.


</details>

**개념 6-16.** LBFGS에서 손실·기울기 계산을 묶는 함수는?  
**답:** `____________`

<details><summary>정답 확인</summary>

closure

**풀이 설명:** 콘텐츠 손실은 생성 이미지와 콘텐츠 이미지의 특징 차이를, 스타일 손실은 Gram Matrix 차이를 측정합니다. 두 항에 가중치를 주어 최적화합니다.


</details>

**개념 6-17.** 콘텐츠·스타일 손실에 곱하는 것은?  
**답:** `____________`

<details><summary>정답 확인</summary>

각각의 가중치

**풀이 설명:** Gram Matrix는 채널별 특징 사이의 상관관계를 계산합니다. 공간 축을 평탄화해 채널×채널 행렬을 만들고 스타일 통계의 차이를 손실로 사용합니다.


</details>

**개념 6-18.** 정규화를 되돌리는 과정은?  
**답:** `____________`

<details><summary>정답 확인</summary>

역정규화

**풀이 설명:** 전처리에서 크기와 채널 정규화는 특징 추출기 학습 조건과 일치해야 합니다. 콘텐츠·스타일·생성 이미지의 텐서 형상도 호환되어야 합니다.


</details>

**개념 6-19.** VGG19의 requires_grad=False 이유는?  
**답:** `____________`

<details><summary>정답 확인</summary>

VGG19 가중치를 학습하지 않기 위해

**풀이 설명:** 사전학습 VGG 특징 추출기의 가중치는 고정하고, 생성 이미지의 픽셀을 최적화 대상으로 두어야 원하는 스타일 전이가 일어납니다.


</details>

**개념 6-20.** torch.bmm가 수행하는 연산은?  
**답:** `____________`

<details><summary>정답 확인</summary>

배치 단위 행렬곱

**풀이 설명:** 이 문제는 신경망 스타일 전이에서 특징 추출, 손실 계산, 최적화 중 해당 단계의 역할을 확인합니다.


</details>


### 풀이
15 : 순전파, 역전파
16 : closure
17 : 가중치
18 : 역정규화(학습 후 이미지 표시를 위해 수행 - \* 배치차원 제거 : squeeze(0), \*채널순서 변환 : BGR -> RGB, \*차원순서변환 : permute(1,2,0).numpy Pytorch : CHW , Matplotlib : HWC )
19 : vgg19 특징 추출목적으로 파라미터 학습방지
20 : 배치 단위 행렬곱(내적)

# 7차시 · 이미지 생성형 AI (GAN·DCGAN·VAE·U-Net)

**지정 노션 원본:** [상위 학습 페이지](https://app.notion.com/p/AI-_7-AI-41e9ba5c1aae80038389eb2c838e7710#f2a9ba5c1aae8231983e810f5d37bfb5) · [코드학습 하위 페이지](https://app.notion.com/p/AI-GAN-DCGAN-VAE-U-Net-0159ba5c1aae83db9bdd01983b5cdeb5) · [내용 빈칸풀이 하위 페이지](https://app.notion.com/p/AI-c929ba5c1aae823daffa01fc3e8ef8fd)

## 7-A. 코드 빈칸 문제

## DCGAN
```python
# DCGAN

# GANGenerator
lat_dimension = 64
chnls = 1
self.inp_sz = 16

self.lin = nn.Linear(lat_dimension, 128*self.inp_sz*self.inp_sz) # (B,128,16,16)
self.up1 = nn.Upsample(scale_factor=2) # (B,128,32, 32)
self.cn1 = nn.Conv2d(128,128,3,stride=1,padding=1) # (B,128, 32, 32)
self.up2 = nn.Upsample(scale_factor=2) # (B,128, 64, 64)
self.cn2 = nn.Conv2d(128,64,3,stride=1,padding=1) # (B,64, 64, 64)
self.cn3 = nn.Conv2d(64,chnls,3,stride=1,padding=1) # (B,1, 64, 64)
self.act = nn.Tanh()  # 출력 [-1,1]

```
```python
#GANDiscriminator
self.disc_model = nn.Sequential(
    *disc_module(chnls,16,bnorm=False),
    *disc_module(16,32),
    *disc_module(32,64),
    *disc_module(64,128)
)
self.adverse_lyr = nn.Sequential(
    nn.Linear(128*(image_sz//16)**2,1), nn.Sigmoid()
)
adv_loss_func = nn.BCELoss()
```

```python
# 판별기: 실제는 1, 생성 이미지는 0에 가깝도록 학습
real_out = disc(real_images)
fake_images = gen(z)
fake_out = disc(fake_images.detach())  # G에 기울기 전달 차단
loss_d = bce(real_out, torch.ones_like(real_out)) + \
         bce(fake_out, torch.zeros_like(fake_out))
optimizer_d.zero_grad()
loss_d.backward()
optimizer_d.step()

# 생성기: D가 가짜를 1로 판단하도록 학습
fake_out = disc(gen(z))
loss_g = bce(fake_out, torch.ones_like(fake_out))
optimizer_g.zero_grad()
loss_g.backward()
optimizer_g.step()
```


### 코드 7-01 · DCGAN Generator 업샘플링

```python
self.lin = nn.Linear(____, 128*16*16)
self.up1 = nn.____(scale_factor=2)
self.cn3 = nn.Conv2d(64, ____, 3, padding=1)
self.act = nn.____()
```

<details><summary>정답·해설 보기</summary>

**정답:** `lat_dimension(64), Upsample, chnls(1), Tanh`  
잠재벡터를 16×16 특징맵으로 변환한 뒤 64×64 흑백 이미지로 확장합니다.

**풀이 설명:** GAN은 생성기 G와 판별기 D를 번갈아 학습합니다. 판별기 갱신 시 가짜 이미지를 detach해 생성기 쪽 기울기를 끊고, 생성기 갱신 시에는 D(G(z))가 진짜로 판정되도록 학습합니다.


</details>

In [ ]:
self.lin = nn.Linear(lat_dimension, 128*16*16)
self.up1 = nn.Upsample(scale_factor=2)
self.cn3 = nn.Conv2d(64,chnls, 3, padding=1)
self.act = nn.Tanh()

### 코드 7-02 · DCGAN 판별기 출력

```python
self.adverse_lyr = nn.Sequential(
    nn.Linear(128*(image_sz//16)**2, ____),
    nn.____()
)
adv_loss_func = nn.____()
```

<details><summary>정답·해설 보기</summary>

**정답:** `1, Sigmoid, BCELoss`  
판별기는 진짜일 확률 하나를 출력합니다.

**풀이 설명:** GAN은 생성기 G와 판별기 D를 번갈아 학습합니다. 판별기 갱신 시 가짜 이미지를 detach해 생성기 쪽 기울기를 끊고, 생성기 갱신 시에는 D(G(z))가 진짜로 판정되도록 학습합니다.


</details>

In [ ]:
self.adverse_lyr = nn.Sequential(
    # 128*(image_sz//16)**2 => flatten 연산
    nn.Linear(128*(image_sz//16)**2, 1), Sigmoid(),
    nn.Tanh()
)
adv_loss_func == nn.BCELoss()

### 코드 7-03 · DCGAN 판별기 학습

```python
fake_images = gen(z)
fake_out = disc(fake_images.____())
loss_d = bce(fake_out, torch.____(fake_out))
optimizer_d.____()
loss_d.backward()
optimizer_d.step()
```

<details><summary>정답·해설 보기</summary>

**정답:** `detach, zeros_like, zero_grad`  
가짜 이미지에 대한 판별기 손실만 표현한 코드입니다. 실제 이미지 손실도 필요합니다.

**풀이 설명:** GAN은 생성기 G와 판별기 D를 번갈아 학습합니다. 판별기 갱신 시 가짜 이미지를 detach해 생성기 쪽 기울기를 끊고, 생성기 갱신 시에는 D(G(z))가 진짜로 판정되도록 학습합니다.


</details>

In [ ]:
fake_images = gen(z) # gen() 함수 가정
fake_out = disc(fake_images.detach()) # disc() 함수 가정
loss_d = bce(fake_out, torch.zeros_like(fake_out))
# 판별기는 가짜를 가짜로 학습목표로 0 라벨링
optimizer_d.zero_grad()
loss_d.backward()
optimizer_d.step()

### 코드 7-04 · DCGAN 생성기 학습

```python
fake_out = disc(gen(z))
loss_g = bce(fake_out, torch.____(fake_out))
optimizer_g.zero_grad()
loss_g.____()
optimizer_g.____()
```

<details><summary>정답·해설 보기</summary>

**정답:** `ones_like, backward, step`  
생성기가 D(G(z))를 1로 판단하도록 유도합니다.

**풀이 설명:** GAN은 생성기 G와 판별기 D를 번갈아 학습합니다. 판별기 갱신 시 가짜 이미지를 detach해 생성기 쪽 기울기를 끊고, 생성기 갱신 시에는 D(G(z))가 진짜로 판정되도록 학습합니다.


</details>

In [ ]:
fake_out = disc(gen(z)) # gen()과 disc() 함수 가정
loss_g = bce(fake_out, torch.ones_like(fake_out))
# 생성기는 판별기가 가짜를 진짜로 유도하기 위해 1 라벨링
optimizer_g.zero_grad()
loss_g.backward()
optimizer_g.step()

## VAE
```python
# VAE 원본 Model.forward 핵심
mean, log_var = self.Encoder(x)
std = torch.exp(0.5 * log_var)  # 로그 분산 → 표준편차
z = self.reparameterization(mean, std)
x_hat = self.Decoder(z)
return x_hat, mean, log_var
```

```python
# VAE Recontruction 과 KL Divergence 손실함수
def loss_function(x, x_hat, mean, log_var):
    # 복원 오차: 원본 x와 복원 x_hat의 차이
    reconstruction = nn.functional.binary_cross_entropy(
        x_hat, x, reduction='sum')
    # KL: 학습된 잠재 분포와 N(0,I)의 차이
    kld = -0.5*torch.sum(1+log_var-mean.pow(2)-log_var.exp())
    return reconstruction + kld

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
```

```python
model.eval()
with torch.no_grad():
    z = torch.randn(16, latent_dim, device=DEVICE)
    generated_images = model.Decoder(z)  # (16,784)
    images = generated_images.view(-1,1,28,28)
```

### 코드 7-05 · VAE 재매개변수화

```python
mean, log_var = self.Encoder(x)
std = torch.____(0.5 * log_var)
epsilon = torch.____(std)
z = mean + std * epsilon
x_hat = self.____(z)
```

<details><summary>정답·해설 보기</summary>

**정답:** `exp, randn_like, Decoder`  
평균과 로그분산에서 표준편차를 얻어 잠재변수 z를 샘플링합니다.

**풀이 설명:** VAE는 평균 μ와 로그분산으로 잠재분포를 표현하고 z=μ+exp(0.5·log_var)·ε로 샘플링합니다. 재구성 손실과 KL 발산을 합쳐 복원 품질과 잠재공간의 규제를 함께 학습합니다.


</details>

In [ ]:
# x : (B, input_dim)
mean,log_var = self.Encoder(x) # Encoder() 가정 : 입력데이터의 잠재분포 반환
std = torch.exp(0.5*log_var) # 로그 분산(log_var)을 표준 편차(std)로 변환
epsilon = torch.randn_like(std) #std 텐서 크기로 N[0,1] 표준정규분포 무작위 노이즈 생성
z = mean + std*epsilon # 재매개변수화 트릭으로 미분가능하여 역전파 가능
x_hat = self.Decoder(z) # 입력과 동일한 형태로 복원
# x_hat : (B, input_dim)

### 코드 7-06 · VAE KL 손실

```python
reconstruction = F.____(x_hat, x, reduction='sum')
kld = -0.5*torch.sum(1 + log_var - mean.____(2) - log_var.____())
loss = reconstruction + kld
```

<details><summary>정답·해설 보기</summary>

**정답:** `binary_cross_entropy, pow, exp`  
재구성 손실과 KL 규제항의 합입니다.

**풀이 설명:** VAE는 평균 μ와 로그분산으로 잠재분포를 표현하고 z=μ+exp(0.5·log_var)·ε로 샘플링합니다. 재구성 손실과 KL 발산을 합쳐 복원 품질과 잠재공간의 규제를 함께 학습합니다.


</details>

In [ ]:
# 복원결과(x_hat) - 원본(x) 계산 >> BCE 적용 ('sum' : 모든 데이터 요소의 손실을 합산)
reconstruction = nn.functional.binary_cross_entropy(x_hat, x, reduction='sum')

# Encoder의 잠재분포N(mean, std**2) - 표준정규분포N(0,1)
# log_var.exp(): 로그 분산을 실제 분산(σ²)으로 변환
# torch.sum(): 배치 및 잠재 차원의 KL 손실 합산
# KL 손실이 작을수록 잠재분포가 표준정규분포에 가까워짐
kld = -0.5*torch.sum(1 + log_var - mean.pow(2) - log_var.exp())

# 복원 정확도와 잠재분포의 정규화를 동시에 최적화
loss = reconstruction + kld


| 손실 | 목적 |
|:---|:---|
| `reconstruction` | 입력 데이터와 복원 데이터의 차이 최소화 |
| `kld` | 잠재분포와 표준정규분포의 차이 최소화 |
| `loss` | 두 손실을 합산하여 Encoder와 Decoder 학습 |


### 코드 7-07 · VAE 배치 처리

```python
x = x.view(x.____(0), -1)
optimizer.____()
x_hat, mean, log_var = model(x)
loss = loss_function(x, x_hat, mean, log_var)
loss.____()
optimizer.____()
```

<details><summary>정답·해설 보기</summary>

**정답:** `size, zero_grad, backward, step`  
마지막 배치 크기가 달라도 처리합니다.

**풀이 설명:** VAE는 평균 μ와 로그분산으로 잠재분포를 표현하고 z=μ+exp(0.5·log_var)·ε로 샘플링합니다. 재구성 손실과 KL 발산을 합쳐 복원 품질과 잠재공간의 규제를 함께 학습합니다.


</details>

In [ ]:
x = x.view(x.size(0), -1) # MNIST 기준: (B, 1, 28, 28) → (B, 784) - 배치크기유지
optimizer.zero_grad()
# VAE 순전파과정 : Encoder (평균, 로그분산) > 재매개변수화() 잠재변수 z 샘플링
# > Decoder(z)=복원데이터 x_hat
x_hat, mean, log_var = model(x)
loss = loss_function(x,x_hat, mean, log_var)
loss.backward()
optimizer.step()

### 코드 7-08 · VAE 새 이미지 생성

```python
model.____()
with torch.____():
    z = torch.____(16, latent_dim, device=DEVICE)
    generated_images = model.____(z)
```

<details><summary>정답·해설 보기</summary>

**정답:** `eval, no_grad, randn, Decoder`  
잠재공간에서 샘플링하여 Decoder로 새 이미지를 생성합니다.

**풀이 설명:** VAE는 평균 μ와 로그분산으로 잠재분포를 표현하고 z=μ+exp(0.5·log_var)·ε로 샘플링합니다. 재구성 손실과 KL 발산을 합쳐 복원 품질과 잠재공간의 규제를 함께 학습합니다.


</details>

In [ ]:
model.eval()
with torch.no_grad():
    z = torch.randn(16, latent_dim, device=DEVICE) # 표준 정규분포 랜덤 노이즈 z 생성
    generated_images = model.Decoder(z)

## 기본 GAN

```python
# 기본 GAN
# 생성기: 100차원 랜덤 벡터 → 784개 이미지 픽셀
 generator = nn.Sequential(
    nn.Linear(latent_dim,256), nn.LeakyReLU(0.2),
    nn.Linear(256,512), nn.LeakyReLU(0.2),
    nn.Linear(512,784), nn.Tanh()
)

# 판별기: 784 픽셀 → 진짜 확률 1개
 discriminator = nn.Sequential(
    nn.Linear(784,512), nn.LeakyReLU(0.2),
    nn.Linear(512,256), nn.LeakyReLU(0.2),
    nn.Linear(256,1), nn.Sigmoid()
)
criterion = nn.BCELoss()
optimizer_G = optim.Adam(generator.parameters(),lr=0.0002,betas=(0.5,0.999))
optimizer_D = optim.Adam(discriminator.parameters(),lr=0.0002,betas=(0.5,0.999))
```

### 코드 7-09 · 기본 GAN Generator

```python
generator = nn.Sequential(
    nn.Linear(latent_dim, 256), nn.LeakyReLU(0.2),
    nn.Linear(256, 512), nn.LeakyReLU(0.2),
    nn.Linear(512, ____), nn.____()
)
```

<details><summary>정답·해설 보기</summary>

**정답:** `784, Tanh`  
28×28 픽셀의 평탄화된 이미지를 생성합니다.

**풀이 설명:** GAN은 생성기 G와 판별기 D를 번갈아 학습합니다. 판별기 갱신 시 가짜 이미지를 detach해 생성기 쪽 기울기를 끊고, 생성기 갱신 시에는 D(G(z))가 진짜로 판정되도록 학습합니다.


</details>

In [ ]:
generator = nn.Sequential(
    nn.Linear(latent_dim, 256), nn.LeakyReLU(0.2), # 잠재벡터 > 256 특징 변환
    nn.Linear(256,512), nn.LeakyReLU(0.2), # 은닉특징 512 차원으로 확장
    nn.Linear(512,784), nn.Tanh() # 최종 이미지 픽셀 28x28=784 개 생성 후 출력 값[-1,1] 제한
)

### 코드 7-10 · 기본 GAN 교대 학습

```python
fake_images = generator(z)
loss_d_fake = criterion(discriminator(fake_images.____()),
                        torch.zeros_like(discriminator(fake_images.detach())))
loss_g = criterion(discriminator(generator(z)),
                   torch.____(discriminator(generator(z))))
```

<details><summary>정답·해설 보기</summary>

**정답:** `detach, ones_like`  
D 학습 때 G로의 기울기를 끊습니다. 중복 forward는 학습용 발췌 그대로입니다.

**풀이 설명:** GAN은 생성기 G와 판별기 D를 번갈아 학습합니다. 판별기 갱신 시 가짜 이미지를 detach해 생성기 쪽 기울기를 끊고, 생성기 갱신 시에는 D(G(z))가 진짜로 판정되도록 학습합니다.


</details>

In [ ]:
fake_images = generator(z)
# 판별기 손실은 가짜를 가짜(0)로 학습
loss_d_fake = criterion(discriminator(fake_images.detach(),
                                      torch.zeros_like(discriminator(fake_images.detach()))))
# 생성기 손실은 D(G(z))를 진짜(1)로 유도
loss_g = criterion(discriminator(generator(z)),
                   torch.ones_like(discriminator9generator(z)))

## U-Net
```python
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels,out_channels,3,padding=1,bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels,out_channels,3,padding=1,bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )
    def forward(self,x):
        return self.double_conv(x)
```

### 코드 7-11 · U-Net DoubleConv

```python
nn.Conv2d(in_channels, out_channels, ____, padding=1, bias=False),
nn.____(out_channels),
nn.____(inplace=True)
```

<details><summary>정답·해설 보기</summary>

**정답:** `3, BatchNorm2d, ReLU`  
3×3 커널과 padding=1로 공간 크기를 유지합니다.

**풀이 설명:** U-Net은 인코더의 고해상도 특징을 skip connection으로 디코더에 전달합니다. 분할 모델의 클래스 logits 출력과 pix2pix 생성기의 RGB/Tanh 출력은 목적이 다릅니다.


</details>

In [1]:
# 3 x 3 커널과 padding -1 로 공간 해상도 유지하며 고수준 특징정보 추출
nn.Conv2d(in_channels, out_channels, 3, padding = 1, bias = False),
nn.BatchNorm2d(out_channels),
nn.ReLU(inplace = True)

NameError: name 'nn' is not defined

### 코드 7-12 · U-Net Skip Connection

```python
x1 = self.____(x1)
x = torch.____([x2, x1], dim=____)
return self.conv(x)
```

<details><summary>정답·해설 보기</summary>

**정답:** `up, cat, 1`  
채널 축을 따라 인코더와 디코더 특징맵을 결합합니다.

**풀이 설명:** U-Net은 인코더의 고해상도 특징을 skip connection으로 디코더에 전달합니다. 분할 모델의 클래스 logits 출력과 pix2pix 생성기의 RGB/Tanh 출력은 목적이 다릅니다.


</details>

In [ ]:
x1 = self.up(x1)
x = torch.cat([x2,x1],dim=1) # 인코더 특징맵 (x2)와 디코더 특징맵(x1)을 채널 방향으로 연결
return self.conv(x)

### 코드 7-13 · pix2pix Generator 출력

```python
self.outc = nn.Sequential(
    nn.Conv2d(64, ____, kernel_size=1),
    nn.____()
)
```

<details><summary>정답·해설 보기</summary>

**정답:** `3, Tanh`  
기본 분할 logits와 달리 RGB 이미지 생성을 위한 출력입니다.

**풀이 설명:** GAN은 생성기 G와 판별기 D를 번갈아 학습합니다. 판별기 갱신 시 가짜 이미지를 detach해 생성기 쪽 기울기를 끊고, 생성기 갱신 시에는 D(G(z))가 진짜로 판정되도록 학습합니다.


</details>

In [ ]:
self.outc = nn.Sequential(
    nn.Conv2d(64, 3, kernel_size = 1),
    nn.Tanh()
)
# 원본 UNetGenerator는 U-Net 본체를 재사용하고
# 마지막 레이어를 Conv2d(64,3,1) → Tanh로 변경합니다.

## 7-B. 개념 빈칸풀이

### 생성형 AI 기초

**개념 7-01.** GAN Generator의 입력과 출력은?  
**답:** `____________`

<details><summary>정답 확인</summary>

잠재벡터 z / 생성 이미지 G(z)

**풀이 설명:** GAN은 생성기 G와 판별기 D를 번갈아 학습합니다. 판별기 갱신 시 가짜 이미지를 detach해 생성기 쪽 기울기를 끊고, 생성기 갱신 시에는 D(G(z))가 진짜로 판정되도록 학습합니다.


</details>

**개념 7-02.** Discriminator가 구분하는 두 대상은?  
**답:** `____________`

<details><summary>정답 확인</summary>

실제 이미지 / 생성(가짜) 이미지

**풀이 설명:** GAN은 생성기 G와 판별기 D를 번갈아 학습합니다. 판별기 갱신 시 가짜 이미지를 detach해 생성기 쪽 기울기를 끊고, 생성기 갱신 시에는 D(G(z))가 진짜로 판정되도록 학습합니다.


</details>

**개념 7-03.** 비슷한 이미지만 반복 생성하는 현상은?  
**답:** `____________`

<details><summary>정답 확인</summary>

Mode Collapse

**풀이 설명:** 이 문제는 생성 모델의 네트워크 구성, 손실 또는 학습 순서를 확인합니다. 각 연산의 입력·출력과 기울기 흐름을 연결해 이해하세요.


</details>

**개념 7-04.** VAE 잠재분포를 표현하는 두 값은?  
**답:** `____________`

<details><summary>정답 확인</summary>

평균 μ / 로그분산 log_var

**풀이 설명:** VAE는 평균 μ와 로그분산으로 잠재분포를 표현하고 z=μ+exp(0.5·log_var)·ε로 샘플링합니다. 재구성 손실과 KL 발산을 합쳐 복원 품질과 잠재공간의 규제를 함께 학습합니다.


</details>

**개념 7-05.** VAE 전체 손실의 두 항은?  
**답:** `____________`

<details><summary>정답 확인</summary>

재구성 손실 / KL 발산

**풀이 설명:** VAE는 평균 μ와 로그분산으로 잠재분포를 표현하고 z=μ+exp(0.5·log_var)·ε로 샘플링합니다. 재구성 손실과 KL 발산을 합쳐 복원 품질과 잠재공간의 규제를 함께 학습합니다.


</details>

###풀이
1 : 잠재공간의 백터 z(확률분포), 생성 이미지 G(z)
2 : 정답이미지 / 생성이미지
3 : mode collapse
4 : 평균, 로그분산(z=μ+exp(0.5·log_var)·ε로 샘플링, z = 잠재공간 확률분포)
5 : Reconstruction, KL Divergence로 재매개변수화 트릭을 사용하여 확률적 샘플링을 역전파 계산에 연결

### DCGAN
```python
# GANGenerator
lat_dimension = 64
chnls = 1
self.inp_sz = 16

self.lin = nn.Linear(lat_dimension, 128*self.inp_sz*self.inp_sz) # (B,128,16,16)
self.up1 = nn.Upsample(scale_factor=2) # (B,128,32, 32)
self.cn1 = nn.Conv2d(128,128,3,stride=1,padding=1) # (B,128, 32, 32)
self.up2 = nn.Upsample(scale_factor=2) # (B,128, 64, 64)
self.cn2 = nn.Conv2d(128,64,3,stride=1,padding=1) # (B,64, 64, 64)
self.cn3 = nn.Conv2d(64,chnls,3,stride=1,padding=1) # (B,1, 64, 64)
self.act = nn.Tanh()  # 출력 [-1,1]

```
```python
#GANDiscriminator
self.disc_model = nn.Sequential(
    *disc_module(chnls,16,bnorm=False),
    *disc_module(16,32),
    *disc_module(32,64),
    *disc_module(64,128)
)
self.adverse_lyr = nn.Sequential(
    nn.Linear(128*(image_sz//16)**2,1), nn.Sigmoid()
)
adv_loss_func = nn.BCELoss()
```

**개념 7-06.** 잠재 차원과 생성 이미지 크기는?  
**답:** `____________`

<details><summary>정답 확인</summary>

64 / 1×64×64

**풀이 설명:** 이 문제는 생성 모델의 네트워크 구성, 손실 또는 학습 순서를 확인합니다. 각 연산의 입력·출력과 기울기 흐름을 연결해 이해하세요.


</details>

**개념 7-07.** 공간 크기를 두 배로 키우는 레이어는?  
**답:** `____________`

<details><summary>정답 확인</summary>

nn.Upsample(scale_factor=2)

**풀이 설명:** U-Net은 인코더의 고해상도 특징을 skip connection으로 디코더에 전달합니다. 분할 모델의 클래스 logits 출력과 pix2pix 생성기의 RGB/Tanh 출력은 목적이 다릅니다.


</details>

**개념 7-08.** 생성기 출력 활성화 함수는?  
**답:** `____________`

<details><summary>정답 확인</summary>

Tanh

**풀이 설명:** GAN은 생성기 G와 판별기 D를 번갈아 학습합니다. 판별기 갱신 시 가짜 이미지를 detach해 생성기 쪽 기울기를 끊고, 생성기 갱신 시에는 D(G(z))가 진짜로 판정되도록 학습합니다.


</details>

**개념 7-09.** 판별기 확률 출력 활성화 함수는?  
**답:** `____________`

<details><summary>정답 확인</summary>

Sigmoid

**풀이 설명:** GAN은 생성기 G와 판별기 D를 번갈아 학습합니다. 판별기 갱신 시 가짜 이미지를 detach해 생성기 쪽 기울기를 끊고, 생성기 갱신 시에는 D(G(z))가 진짜로 판정되도록 학습합니다.


</details>

**개념 7-10.** 판별기 학습 중 생성기로의 기울기를 끊는 메서드는?  
**답:** `____________`

<details><summary>정답 확인</summary>

detach()

**풀이 설명:** GAN은 생성기 G와 판별기 D를 번갈아 학습합니다. 판별기 갱신 시 가짜 이미지를 detach해 생성기 쪽 기울기를 끊고, 생성기 갱신 시에는 D(G(z))가 진짜로 판정되도록 학습합니다.


</details>

### VAE


| 항목 | 원본 설정 |
|:---|:---|
| 데이터 | MNIST, 28×28 흑백 |
| 입력 차원 | `x_dim=784` |
| 은닉 차원 | `hidden_dim=400` |
| 잠재 차원 | `latent_dim=200` |
| 배치 | `batch_size=100` |
| 학습률 | `lr=1e-3` |
| 에폭 | `epochs=30` |


**개념 7-11.** 28×28 MNIST를 평탄화한 차원은?  
**답:** `____________`

<details><summary>정답 확인</summary>

784

**풀이 설명:** 이 문제는 생성 모델의 네트워크 구성, 손실 또는 학습 순서를 확인합니다. 각 연산의 입력·출력과 기울기 흐름을 연결해 이해하세요.


</details>

**개념 7-12.** 원본 VAE 잠재벡터 차원은?  
**답:** `____________`

<details><summary>정답 확인</summary>

200

**풀이 설명:** VAE는 평균 μ와 로그분산으로 잠재분포를 표현하고 z=μ+exp(0.5·log_var)·ε로 샘플링합니다. 재구성 손실과 KL 발산을 합쳐 복원 품질과 잠재공간의 규제를 함께 학습합니다.


</details>

**개념 7-13.** z = μ + ____ × ε  
**답:** `____________`

<details><summary>정답 확인</summary>

exp(0.5 × log_var)

**풀이 설명:** VAE는 평균 μ와 로그분산으로 잠재분포를 표현하고 z=μ+exp(0.5·log_var)·ε로 샘플링합니다. 재구성 손실과 KL 발산을 합쳐 복원 품질과 잠재공간의 규제를 함께 학습합니다.


</details>

**개념 7-14.** binary_cross_entropy의 reduction 값은?  
**답:** `____________`

<details><summary>정답 확인</summary>

'sum'

**풀이 설명:** 이 문제는 생성 모델의 네트워크 구성, 손실 또는 학습 순서를 확인합니다. 각 연산의 입력·출력과 기울기 흐름을 연결해 이해하세요.


</details>

**개념 7-15.** 배치 크기를 유지하는 view 첫 번째 인자는?  
**답:** `____________`

<details><summary>정답 확인</summary>

x.size(0)

**풀이 설명:** 이 문제는 생성 모델의 네트워크 구성, 손실 또는 학습 순서를 확인합니다. 각 연산의 입력·출력과 기울기 흐름을 연결해 이해하세요.


</details>

###풀이
6 : 잠재차원 : lat_dimension = 64, 생성이미지 : chnls x late_dimension x late_dimension = 1×64×64
7 : nn.Upsample(scale_factor=2)
8 : nn.Tanh()
9 : nn.Sigmoid()
10 : detach()

###풀이
11 : x_dim (=28x28) = 784
12 : latent_dim = 200
13 : exp(0.5 × log_var)  (\* 로그 분산 → 표준편차, \*재매개변수화 트릭 : z= mean+std*epsilon)
14 : 'sum'(복원오차 : 원본 x - 복원 x_hat)
15 : x.size(0)

### 기본 GAN
```python
import torch.nn as nn
latent_dim = 100

# 생성기: 100차원 랜덤 벡터 → 784개 이미지 픽셀
 generator = nn.Sequential(
    nn.Linear(latent_dim,256), nn.LeakyReLU(0.2),
    nn.Linear(256,512), nn.LeakyReLU(0.2),
    nn.Linear(512,784), nn.Tanh()
)

# 판별기: 784 픽셀 → 진짜 확률 1개
 discriminator = nn.Sequential(
    nn.Linear(784,512), nn.LeakyReLU(0.2),
    nn.Linear(512,256), nn.LeakyReLU(0.2),
    nn.Linear(256,1), nn.Sigmoid()
)
criterion = nn.BCELoss()
optimizer_G = optim.Adam(generator.parameters(),lr=0.0002,betas=(0.5,0.999))
optimizer_D = optim.Adam(discriminator.parameters(),lr=0.0002,betas=(0.5,0.999))
```

**개념 7-16.** 잠재벡터 길이와 생성 이미지 픽셀 수는?  
**답:** `____________`

<details><summary>정답 확인</summary>

100 / 784

**풀이 설명:** 이 문제는 생성 모델의 네트워크 구성, 손실 또는 학습 순서를 확인합니다. 각 연산의 입력·출력과 기울기 흐름을 연결해 이해하세요.


</details>

**개념 7-17.** 기본 GAN 손실함수는?  
**답:** `____________`

<details><summary>정답 확인</summary>

BCELoss

**풀이 설명:** GAN은 생성기 G와 판별기 D를 번갈아 학습합니다. 판별기 갱신 시 가짜 이미지를 detach해 생성기 쪽 기울기를 끊고, 생성기 갱신 시에는 D(G(z))가 진짜로 판정되도록 학습합니다.


</details>

**개념 7-18.** 원본 GAN 학습률은?  
**답:** `____________`

<details><summary>정답 확인</summary>

0.0002 (2e-4)

**풀이 설명:** GAN은 생성기 G와 판별기 D를 번갈아 학습합니다. 판별기 갱신 시 가짜 이미지를 detach해 생성기 쪽 기울기를 끊고, 생성기 갱신 시에는 D(G(z))가 진짜로 판정되도록 학습합니다.


</details>

**개념 7-19.** D 학습 시 가짜 이미지 정답 라벨은?  
**답:** `____________`

<details><summary>정답 확인</summary>

0

**풀이 설명:** 이 문제는 생성 모델의 네트워크 구성, 손실 또는 학습 순서를 확인합니다. 각 연산의 입력·출력과 기울기 흐름을 연결해 이해하세요.


</details>

**개념 7-20.** G 학습 시 D(G(z)) 목표 라벨은?  
**답:** `____________`

<details><summary>정답 확인</summary>

1

**풀이 설명:** 이 문제는 생성 모델의 네트워크 구성, 손실 또는 학습 순서를 확인합니다. 각 연산의 입력·출력과 기울기 흐름을 연결해 이해하세요.


</details>

###풀이
16 : latent_dim = 100, 784
17 : nn.BCELoss()
18 : 0.0002
19 : 0 (판별기는 가짜를 가짜로 학습목표)
20 : 1 (생성기는 가짜를 진짜로 판단하도록 유도, *생성기 학습시에는 판별기 가중치 변경 없이 생성기의 가중치만 업데이트)

### U-Net·pix2pix

| 단계 | 입력→출력 채널 | 256×256 기준 해상도 |
|:---|:---:|:---:|
| inc | 3→64 | 256×256 |
| down1 | 64→128 | 128×128 |
| down2 | 128→256 | 64×64 |
| down3 | 256→512 | 32×32 |
| down4 | 512→1024 | 16×16 |
| up1 | 1024→512 | 32×32 |
| up2 | 512→256 | 64×64 |
| up3 | 256→128 | 128×128 |
| up4 | 128→64 | 256×256 |
| outc | 64→2 | 256×256 |

```python
# 기본 U-Net 출력
self.outc = nn.Conv2d(64,n_classes,kernel_size=1)

# pix2pix 생성기용 변경
self.outc = nn.Sequential(
    nn.Conv2d(64,3,kernel_size=1),
    nn.Tanh()  # RGB 출력 [-1,1]
)
```

**개념 7-21.** 인코더와 디코더를 연결하는 구조는?  
**답:** `____________`

<details><summary>정답 확인</summary>

Skip Connection

**풀이 설명:** U-Net은 인코더의 고해상도 특징을 skip connection으로 디코더에 전달합니다. 분할 모델의 클래스 logits 출력과 pix2pix 생성기의 RGB/Tanh 출력은 목적이 다릅니다.


</details>

**개념 7-22.** Down 블록 해상도 축소 연산은?  
**답:** `____________`

<details><summary>정답 확인</summary>

MaxPool2d(2)

**풀이 설명:** U-Net은 인코더의 고해상도 특징을 skip connection으로 디코더에 전달합니다. 분할 모델의 클래스 logits 출력과 pix2pix 생성기의 RGB/Tanh 출력은 목적이 다릅니다.


</details>

**개념 7-23.** torch.cat 결합 차원 dim은?  
**답:** `____________`

<details><summary>정답 확인</summary>

1 (채널)

**풀이 설명:** 이 문제는 생성 모델의 네트워크 구성, 손실 또는 학습 순서를 확인합니다. 각 연산의 입력·출력과 기울기 흐름을 연결해 이해하세요.


</details>

**개념 7-24.** 기본 U-Net 출력 채널 수와 의미는?  
**답:** `____________`

<details><summary>정답 확인</summary>

2 / 분할 클래스 logits

**풀이 설명:** U-Net은 인코더의 고해상도 특징을 skip connection으로 디코더에 전달합니다. 분할 모델의 클래스 logits 출력과 pix2pix 생성기의 RGB/Tanh 출력은 목적이 다릅니다.


</details>

**개념 7-25.** pix2pix Generator 출력 채널과 활성화 함수는?  
**답:** `____________`

<details><summary>정답 확인</summary>

3(RGB) / Tanh

**풀이 설명:** GAN은 생성기 G와 판별기 D를 번갈아 학습합니다. 판별기 갱신 시 가짜 이미지를 detach해 생성기 쪽 기울기를 끊고, 생성기 갱신 시에는 D(G(z))가 진짜로 판정되도록 학습합니다.


</details>

###풀이
21 : Skip Connection
22 : MaxPool2d(2) (*H,W는 절반,채널은 증가)
23 : torch.cat([x2,x1], dim = 1) (*인코더 skip과 채녈 연결시)
24 : outc단계 2 출력 : 분할 클래스 logits
25 : 3(RGB) / Tanh

# 8차시 · 이미지 캡셔닝 (CNN·LSTM)

**지정 노션 원본:** [상위 학습 페이지](https://app.notion.com/p/41e9ba5c1aae80449664de34fc2748a8) · [코드학습 하위 페이지](https://app.notion.com/p/41e9ba5c1aae80d78b72c461a234f538) · [내용 빈칸풀이 하위 페이지](https://app.notion.com/p/41e9ba5c1aae8009854dd4e84e6b4b02)



원본의 코드학습 페이지는 주로 실습 흐름과 그룹스터디 질문으로 구성되어 있어, 아래의 **코드 연습 5문제는 원본 학습 내용을 바탕으로 새로 구성한 연습문제**입니다.

## 8-A. 코드학습 · 그룹스터디 5문제

### 코드학습 8-01

**문제:** 이미지 특징벡터 256차원은 어디에서 만들어지는가?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

CNN 인코더의 특징 추출 및 선형 투영 단계에서 만들어집니다.

**풀이 설명:** CNN 인코더는 이미지에서 특징을 추출하고, 선형 투영 등을 거쳐 LSTM이 받을 임베딩 크기로 맞춥니다. 이 단계는 이미지 정보와 단어 시퀀스 모델을 연결합니다.


</details>

1 : CNN model ResNet 기반 인코더, embedding_size=256

### 코드학습 8-02

**문제:** CNN 출력과 LSTM 입력 차원을 맞추는 이유는?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

디코더가 받는 특징 벡터의 마지막 차원이 LSTM 입력 차원과 일치해야 연산할 수 있습니다.

**풀이 설명:** CNN 인코더는 이미지에서 특징을 추출하고, 선형 투영 등을 거쳐 LSTM이 받을 임베딩 크기로 맞춥니다. 이 단계는 이미지 정보와 단어 시퀀스 모델을 연결합니다.


</details>

2 : LSTM은 CNN에서 추출한 이미지 특징 벡터를 기반으로 문장을 생성한다. CNN의 특징벡터 출력은 LSTM 입력차원과 동일해야한다.

### 코드학습 8-03
```python
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

def collate_function(batch):
    images, captions = zip(*batch)

    # 이미지들을 배치 텐서로 결합
    images = torch.stack(images, dim=0)

    # 각 캡션의 실제 길이 저장 (PAD 제외)
    lengths = [len(cap) for cap in captions]

    # 가장 긴 캡션에 맞춰 PAD 추가
    captions = pad_sequence(
        captions,
        batch_first=True,
        padding_value=0
    )

    return images, captions, lengths
```
**문제:** collate_function의 padding과 lengths가 필요한 이유는?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

가변 길이 캡션을 동일 배치 텐서로 구성하고 실제 시퀀스 길이를 보존하기 위해서입니다.

**풀이 설명:** 캡션 길이가 서로 다르면 하나의 배치 텐서로 묶기 위해 패딩이 필요합니다. 실제 길이를 함께 전달하면 패딩 토큰을 손실·시퀀스 처리에서 적절히 제외할 수 있습니다.


</details>

3 : 배치 단위 학습이 가능하도록 길이가 다른 캡션들을 일정한 길이로 유지해주어야 한다. 이 과정에서 lenghts는 가장 긴 캡션의 길이를 선택하고, 캡션마다 padding값을 설정하여 연산시 불필요한 학습을 방지하는 구분자 역할을 할 수 있다.

### 코드학습 8-04

**문제:** Teacher Forcing의 학습 입력과 생성 추론 입력은 어떻게 다른가?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

학습에서는 이전 정답 단어를, 생성 추론에서는 직전 예측 단어를 사용합니다.

**풀이 설명:** LSTM 디코더는 이전 단어와 은닉 상태를 바탕으로 다음 단어를 예측합니다. 학습의 Teacher Forcing은 정답 이전 단어를 사용하지만, 추론은 이전 예측 단어를 다시 입력합니다.


</details>

4 : 학습시엔 정답 단어를 입력하고, 생성 추론에는 start부터 end까지 문장을 순차적으로 생성한다.

### 코드학습 8-05

**문제:** BLEU·CIDEr가 높아도 문장이 어색할 수 있는 이유는?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

참조 문장과의 통계적 유사성이 문법·사실성·자연스러움을 완전히 보장하지 않기 때문입니다.

**풀이 설명:** BLEU는 n-gram 일치도를, METEOR는 어간·유의어와 재현율 등을, CIDEr는 TF-IDF 가중 n-gram 합의도를 사용합니다. 수치가 높아도 자연스러움이나 이미지와의 사실적 일치를 완전히 보장하지는 않습니다.


</details>

5 : 두 지표 공통으로 사용하는 Ngram 일치도 중심의 평가는 문맥의 자연스러움을 충분히 반영하지 못한다

## 8-B. 내용 빈칸풀이 (20문제)

### 개념 8-01

**문제:** CNN Encoder의 목적은?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

이미지의 시각 특징을 추출해 LSTM에 전달할 특징 벡터로 변환

**풀이 설명:** CNN 인코더는 이미지에서 특징을 추출하고, 선형 투영 등을 거쳐 LSTM이 받을 임베딩 크기로 맞춥니다. 이 단계는 이미지 정보와 단어 시퀀스 모델을 연결합니다.


</details>

1 : 초기 합성곱층은 에지와 경계, 중간층은 질감, 깊은 층은 사물의 형태등 이미지 특징을 추출하고 Decoder에 전달할 특징 벡터이 형태로 반환한다

### 개념 8-02

**문제:** LSTM Decoder가 순차적으로 예측하는 것은?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

다음 단어의 확률 분포

**풀이 설명:** LSTM 디코더는 이전 단어와 은닉 상태를 바탕으로 다음 단어를 예측합니다. 학습의 Teacher Forcing은 정답 이전 단어를 사용하지만, 추론은 이전 예측 단어를 다시 입력합니다.


</details>

2 : 이미지 특징과 이전 단어를 조건으로 다음 단어의 확률 분포를 예측한다.

### 개념 8-03

**문제:** pad 토큰이 필요한 이유는?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

가변 길이 캡션을 배치 내 동일 길이로 맞추기 위해

**풀이 설명:** 캡션 길이가 서로 다르면 하나의 배치 텐서로 묶기 위해 패딩이 필요합니다. 실제 길이를 함께 전달하면 패딩 토큰을 손실·시퀀스 처리에서 적절히 제외할 수 있습니다.


</details>

3 : 서로 다른 이미지 캡션을 동일한 연산의 길이로 만들어 배치단위 학습 형태를 만들고 pad 위치를 제외하여 불필요한 학습을 방지할 수 있다

### 개념 8-04

**문제:** start/end 토큰의 역할은?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

문장 생성 시작 / 종료

**풀이 설명:** LSTM 디코더는 이전 단어와 은닉 상태를 바탕으로 다음 단어를 예측합니다. 학습의 Teacher Forcing은 정답 이전 단어를 사용하지만, 추론은 이전 예측 단어를 다시 입력합니다.


</details>

4 : 문장의 시작과 끝을 구분하는 역할

### 개념 8-05

**문제:** Teacher Forcing에서 입력으로 쓰는 것은?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

이전 시점의 정답 단어

**풀이 설명:** LSTM 디코더는 이전 단어와 은닉 상태를 바탕으로 다음 단어를 예측합니다. 학습의 Teacher Forcing은 정답 이전 단어를 사용하지만, 추론은 이전 예측 단어를 다시 입력합니다.


</details>

5 : 학습시 이전 정답 단어를 입력받는다

### 개념 8-06

**문제:** CrossEntropyLoss가 비교하는 것은?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

다음 단어 logits / 정답 단어 인덱스

**풀이 설명:** LSTM 디코더는 이전 단어와 은닉 상태를 바탕으로 다음 단어를 예측합니다. 학습의 Teacher Forcing은 정답 이전 단어를 사용하지만, 추론은 이전 예측 단어를 다시 입력합니다.


</details>

6 : 다음 단어 logits과 목표 인덱스의 차이

### 개념 8-07

**문제:** Greedy Search의 선택 기준은?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

매 시점 확률이 가장 높은 단어

**풀이 설명:** LSTM 디코더는 이전 단어와 은닉 상태를 바탕으로 다음 단어를 예측합니다. 학습의 Teacher Forcing은 정답 이전 단어를 사용하지만, 추론은 이전 예측 단어를 다시 입력합니다.


</details>

7 : 시점별 최고 확률 단어를 선택

### 개념 8-08

**문제:** Beam Search의 k는?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

유지할 상위 후보 문장 경로 수

**풀이 설명:** Greedy는 각 단계의 최고 확률 단어를 즉시 선택하고 Beam Search는 여러 후보 경로를 유지합니다. Attention은 단어 생성 시 관련 이미지 위치에 더 높은 가중치를 둡니다.


</details>

8 : 상위 확률 k개 경로

### 개념 8-09

**문제:** Attention의 역할은?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

생성 단어별로 관련 이미지 특징 위치에 가중치 부여

**풀이 설명:** LSTM 디코더는 이전 단어와 은닉 상태를 바탕으로 다음 단어를 예측합니다. 학습의 Teacher Forcing은 정답 이전 단어를 사용하지만, 추론은 이전 예측 단어를 다시 입력합니다.


</details>

9 : 단어마다 이미지 특징맵의 관련 영역에 동적 가중치 부여

### 개념 8-10

**문제:** BLEU의 비교 기본 단위는?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

n-gram

**풀이 설명:** BLEU는 n-gram 일치도를, METEOR는 어간·유의어와 재현율 등을, CIDEr는 TF-IDF 가중 n-gram 합의도를 사용합니다. 수치가 높아도 자연스러움이나 이미지와의 사실적 일치를 완전히 보장하지는 않습니다.


</details>

10 : ngram 정밀도

### 개념 8-11

**문제:** METEOR가 추가 고려하는 요소는?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

어간·유의어 매칭과 정밀도·재현율

**풀이 설명:** BLEU는 n-gram 일치도를, METEOR는 어간·유의어와 재현율 등을, CIDEr는 TF-IDF 가중 n-gram 합의도를 사용합니다. 수치가 높아도 자연스러움이나 이미지와의 사실적 일치를 완전히 보장하지는 않습니다.


</details>

11 : 재현율과 어간/유의어 매칭

### 개념 8-12

**문제:** CIDEr의 가중치는?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

TF-IDF 기반 단어·n-gram 가중치

**풀이 설명:** LSTM 디코더는 이전 단어와 은닉 상태를 바탕으로 다음 단어를 예측합니다. 학습의 Teacher Forcing은 정답 이전 단어를 사용하지만, 추론은 이전 예측 단어를 다시 입력합니다.


</details>

12 : TF-IDF 기반 다중 참조 캡션 합의도

### 개념 8-13

**문제:** Flickr8k Encoder는?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

사전학습 ResNet-18

**풀이 설명:** CNN 인코더는 이미지에서 특징을 추출하고, 선형 투영 등을 거쳐 LSTM이 받을 임베딩 크기로 맞춥니다. 이 단계는 이미지 정보와 단어 시퀀스 모델을 연결합니다.


</details>

13 : ResNet18

### 개념 8-14

**문제:** Flickr8k embedding/hidden 크기는?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

256 / 512

**풀이 설명:** CNN 인코더는 이미지에서 특징을 추출하고, 선형 투영 등을 거쳐 LSTM이 받을 임베딩 크기로 맞춥니다. 이 단계는 이미지 정보와 단어 시퀀스 모델을 연결합니다.


</details>

14 : embedding = 256, hidden = 512

### 개념 8-15

**문제:** CIDEr 보충 코드의 패키지는?

**답:** `____________`

<details><summary>정답·해설 보기</summary>

pycocoevalcap

**풀이 설명:** BLEU는 n-gram 일치도를, METEOR는 어간·유의어와 재현율 등을, CIDEr는 TF-IDF 가중 n-gram 합의도를 사용합니다. 수치가 높아도 자연스러움이나 이미지와의 사실적 일치를 완전히 보장하지는 않습니다.


</details>

15 : pycocoevalcap
```python
!pip install -q pycocoevalcap
from pycocoevalcap.cider.cider import Cider
```